# 02 · Татарская F5-TTS: короткий референс → новая речь без обучения

**Цель:** проверить татарскую F5-TTS `yasalma/f5-tts-finetuned-tt` с коротким **разрешённым** образцом голоса. Само дообучение не требуется. Это эксперимент: модель мало документирована, словарь может оказаться несовместим, а контрольная точка крупная (~5.4 GB на диске).

Для работы: `MyDrive/tatar_audio_lab/authorized_reference/reference.wav` — 5–12 секунд чистого голоса одного разрешившего человека; `reference.txt` — **дословная татарская транскрипция**, точно соответствующая WAV. Можно использовать свой голос или голос отца с его осознанного согласия. Не использовать чужие публичные исполнения для создания узнаваемого клона без разрешений.

In [ ]:
import sys, subprocess, shutil
from pathlib import Path
import torch
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'нет')
assert torch.cuda.is_available(), 'Для 5GB модели рекомендуется включить GPU: Runtime > Change runtime type > T4 GPU'
subprocess.run([sys.executable,'-m','pip','install','-q',
                'git+https://github.com/SWivid/F5-TTS.git', 'huggingface_hub>=0.26',
                'soundfile>=0.12', 'numpy'],check=True)


In [ ]:
from google.colab import drive
from IPython.display import Audio, display
from huggingface_hub import hf_hub_download
import soundfile as sf
import numpy as np
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/tatar_audio_lab')
REF_DIR=ROOT/'authorized_reference'
REF_DIR.mkdir(parents=True,exist_ok=True)
OUT=ROOT/'f5_generated'; OUT.mkdir(parents=True,exist_ok=True)
REF_WAV=REF_DIR/'reference.wav'
REF_TXT=REF_DIR/'reference.txt'
print('Скопируйте разрешённый короткий wav сюда:',REF_WAV)
print('Его дословный текст сюда:',REF_TXT)


In [ ]:
VOICE_RIGHTS_CONFIRMED = False  # выставить True только при необходимом разрешении
assert VOICE_RIGHTS_CONFIRMED, 'Сначала подтвердите права на создание и использование синтетического голоса.'
assert REF_WAV.exists() and REF_TXT.exists(), 'Нет разрешённого WAV и/или дословного текста.'
reference_text=REF_TXT.read_text(encoding='utf-8').strip()
assert len(reference_text)>10, 'Слишком короткая референсная транскрипция'
info=sf.info(str(REF_WAV))
assert 3<=info.duration<=12.5, f'Длина референса должна быть около 5–12 сек, сейчас {info.duration:.1f}'
print('Референс:',reference_text,'длина',info.duration)
display(Audio(filename=str(REF_WAV)))


**Внимание:** checkpoint `yasalma/f5-tts-finetuned-tt` опубликован как `.pt` и не имеет полноценной инструкции по версии словаря; загрузку и произношение нужно экспериментально проверить. Если `F5TTS` выдаст ошибку несовместимости, **не** подменяйте контрольную точку случайной английской моделью — результат не будет проверкой татарского TTS.

In [ ]:
CKPT_ID='yasalma/f5-tts-finetuned-tt'
CKPT_FILENAME='model_136000.pt'
ckpt=hf_hub_download(repo_id=CKPT_ID,filename=CKPT_FILENAME)
print('Скачана checkpoint:',ckpt)
from f5_tts.api import F5TTS
f5=F5TTS(model='F5TTS_v1_Base',ckpt_file=str(ckpt),device='cuda')
print('F5 татарская модель загружена.')


In [ ]:
TEST_SENTENCES=[
    'Бүген иртән безнең тәрәзә төбенә кечкенә болыт килеп кунды.',
    'Чәйнек кинәт ялга китәргә булды, ә чынаяклар аңа бик гаҗәпләнеп карадылар.',
    'Кыз ишекне ачты да, әкрен генә: «Исәнме, диңгез», — диде.'
]
for i,txt in enumerate(TEST_SENTENCES,1):
    out=OUT/f'variant_{i:02d}.wav'
    wav,sr,spec=f5.infer(ref_file=str(REF_WAV),ref_text=reference_text,
                          gen_text=txt, file_wave=str(out),seed=42+i,
                          nfe_step=32,speed=1.0)
    print(out, 'частота',sr,'текст:',txt)
    display(Audio(filename=str(out)))


### Генерация целой сказки

Положите текст истории в `MyDrive/tatar_audio_lab/story_tt.txt` (UTF-8). Сегменты около 100–130 символов, чтобы не превышать лимит длины F5-TTS с учётом референса. Каждый фрагмент озвучивается отдельно и соединяется мягкими паузами; ритм и произношение всё равно нужно слушать.

In [ ]:
import re
STORY=ROOT/'story_tt.txt'
if STORY.exists():
    story=STORY.read_text(encoding='utf-8').strip()
    sentences=re.split(r'(?<=[.!?…])\s+',story)
    snippets=[]
    buf=''
    for s in sentences:
        for piece in [s[k:k+120] for k in range(0,len(s),120)]:
            if buf and len(buf)+len(piece)+1>130:
                snippets.append(buf); buf=''
            buf=(buf+' '+piece).strip()
    if buf: snippets.append(buf)
    generated=[]; SR=None
    for j,s in enumerate(snippets,1):
        wav,sr,_=f5.infer(ref_file=str(REF_WAV),ref_text=reference_text,gen_text=s,
                         seed=100+j,nfe_step=32,speed=1.0)
        SR=sr
        generated.append(wav)
    silence=np.zeros(int(SR*0.32),dtype=np.float32)
    mixed=np.concatenate([x for w in generated for x in (w,silence)])
    sf.write(str(OUT/'story_f5_tt.wav'),mixed,SR)
    print('Полный WAV:', OUT/'story_f5_tt.wav')
else:
    print('Текст story_tt.txt не найден; три контрольные реплики уже доступны выше.')


**После теста:** оцените татарские звуки, читаемость окончаний, интонации и стабильность голоса на нескольких репликах. Сравните с Silero и стандартным татарским Piper. Если модель не поддерживает буквы ә/ө/ү/җ/ң/һ должным образом, результат нельзя выпускать.